# 光明翻译器 — ERNIE-4.5-0.3B LoRA 微调 Notebook

本 Notebook 将 ERNIE-4.5-0.3B 微调为 **Python → 光明 v3.2 代码翻译器**。

每个 Cell 对应一个独立步骤，可以逐步执行、调试和观察。

**前置条件**：
- GPU 环境（推荐 T4 / 4060 及以上，4GB+ 显存）
- PaddlePaddle GPU 版已安装
- 训练数据 `sft_dataset.jsonl` 已生成

**完整文档**：`tools/ai_copilot/README_SFT.md`

## 第 1 步：环境准备

In [ ]:
# 安装依赖（首次运行时取消注释）
# !pip install --upgrade --pre paddlenlp
# !pip install --upgrade aistudio-sdk
# !pip install paddlepaddle-gpu  # GPU 版本

In [ ]:
# 验证环境
import paddle
print(f"PaddlePaddle: {paddle.__version__}")
print(f"GPU 可用: {paddle.is_compiled_with_cuda()}")
if paddle.is_compiled_with_cuda():
    print(f"设备: {paddle.device.get_device()}")

import paddlenlp
print(f"PaddleNLP: {paddlenlp.__version__}")

import os
import json
print("\n环境检查通过 ✓")

## 第 2 步：检查训练数据

In [ ]:
# 检查训练数据文件
DATASET_PATH = "sft_dataset.jsonl"

if not os.path.isfile(DATASET_PATH):
    print("训练数据不存在，正在生成...")
    !python build_sft_dataset.py

# 统计数据
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f"总条数: {len(lines)}")

# 按类别统计
categories = {}
for line in lines:
    d = json.loads(line)
    cat = d.get('category', 'unknown')
    categories[cat] = categories.get(cat, 0) + 1

print("\n按类别:")
for cat, count in sorted(categories.items(), key=lambda x: -x[1]):
    print(f"  {cat:8s} {count:4d} 条")

In [ ]:
# 抽查几条数据
import random
samples = random.sample(lines, min(3, len(lines)))
for s in samples:
    d = json.loads(s)
    print(f"[{d['category']}]")
    print(f"  指令: {d['instruction']}")
    print(f"  输入: {d['input'][:80]}")
    print(f"  输出: {d['output'][:80]}")
    print()

## 第 3 步：下载预训练模型

In [ ]:
# 设置模型路径
MODEL_NAME = "paddlepaddle/ernie-4.5-0.3b"
MODEL_DIR = "./model_cache/ernie-4.5-0.3b"

# 下载模型（如果尚未下载）
if not os.path.isdir(MODEL_DIR) or not os.path.isfile(os.path.join(MODEL_DIR, 'config.json')):
    print(f"下载模型: {MODEL_NAME}")
    !aistudio download --model {MODEL_NAME} --target {MODEL_DIR}
else:
    print(f"模型已存在: {MODEL_DIR}")

# 验证模型文件
model_files = os.listdir(MODEL_DIR)
print(f"模型文件数: {len(model_files)}")
print(f"包含 config.json: {'config.json' in model_files}")

## 第 4 步：配置 LoRA 训练参数

关键参数说明：
- `lora_rank`: LoRA 秩，越大效果越好但显存占用更多（推荐 16-32）
- `epochs`: 训练轮数（推荐 3-5）
- `lr`: 学习率（推荐 1e-4 ~ 2e-4）
- `batch_size`: 批大小（4GB 显存用 2-4）

In [ ]:
# ═══ 训练超参数（可调节）═══
LORA_RANK = 16         # LoRA 秩
LORA_ALPHA = 32        # LoRA alpha（通常 = 2 × rank）
EPOCHS = 3             # 训练轮数
BATCH_SIZE = 4         # 批大小
GRAD_ACCUM = 4         # 梯度累积步数
LEARNING_RATE = 1e-4   # 学习率
MAX_SEQ_LEN = 1024     # 最大序列长度
WARMUP_RATIO = 0.05    # 预热比例
SEED = 42              # 随机种子

OUTPUT_DIR = "./output/light_translator"

print("训练参数:")
print(f"  LoRA rank={LORA_RANK}, alpha={LORA_ALPHA}")
print(f"  epochs={EPOCHS}, batch_size={BATCH_SIZE}, grad_accum={GRAD_ACCUM}")
print(f"  lr={LEARNING_RATE}, max_seq_len={MAX_SEQ_LEN}")
print(f"  有效批大小: {BATCH_SIZE * GRAD_ACCUM}")

## 第 5 步：生成训练配置并启动训练

In [ ]:
# 生成 ERNIEKit 训练配置
import yaml

os.makedirs(OUTPUT_DIR, exist_ok=True)

config = {
    'model': {
        'model_name_or_path': MODEL_DIR,
        'dtype': 'bfloat16',
    },
    'method': {
        'name': 'lora',
        'lora_rank': LORA_RANK,
        'lora_alpha': LORA_ALPHA,
        'lora_dropout': 0.05,
        'target_modules': ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    },
    'data': {
        'train_datasets': [{
            'name': 'light_sft',
            'data_source': os.path.abspath(DATASET_PATH),
            'format': 'jsonl',
            'columns': {
                'prompt': 'instruction',
                'query': 'input',
                'response': 'output',
            },
        }],
    },
    'training': {
        'output_dir': os.path.join(OUTPUT_DIR, 'checkpoints'),
        'per_device_train_batch_size': BATCH_SIZE,
        'gradient_accumulation_steps': GRAD_ACCUM,
        'num_train_epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'warmup_ratio': WARMUP_RATIO,
        'logging_steps': 10,
        'save_steps': 100,
        'save_total_limit': 3,
        'max_seq_length': MAX_SEQ_LEN,
        'bf16': True,
        'fp16': False,
        'gradient_checkpointing': True,
        'seed': SEED,
    },
}

yaml_path = os.path.join(OUTPUT_DIR, 'train_config.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(config, f, allow_unicode=True, default_flow_style=False)

print(f"配置文件已生成: {yaml_path}")

In [ ]:
# 启动训练
print("开始 LoRA SFT 训练...")
print(f"命令: erniekit train {yaml_path}")
print()

!erniekit train {yaml_path}

## 第 6 步：合并 LoRA 权重

In [ ]:
# 合并 LoRA 权重到基础模型
MERGED_DIR = os.path.join(OUTPUT_DIR, 'merged')
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')

# 找到最新的 checkpoint
if os.path.isdir(CHECKPOINT_DIR):
    checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint')]
    if checkpoints:
        latest_ckpt = os.path.join(CHECKPOINT_DIR, sorted(checkpoints)[-1])
        print(f"最新 checkpoint: {latest_ckpt}")
        print(f"\n合并命令:")
        print(f"erniekit merge --model {MODEL_DIR} --lora {latest_ckpt} --output {MERGED_DIR}")
        !erniekit merge --model {MODEL_DIR} --lora {latest_ckpt} --output {MERGED_DIR}
    else:
        print("未找到 checkpoint，请检查训练是否成功")
else:
    print(f"checkpoint 目录不存在: {CHECKPOINT_DIR}")

## 第 7 步：测试微调后的模型

In [ ]:
# 加载合并后的模型进行推理测试
from paddlenlp.transformers import AutoTokenizer, AutoModelForCausalLM

test_model_path = MERGED_DIR if os.path.isdir(MERGED_DIR) else MODEL_DIR
print(f"加载模型: {test_model_path}")

tokenizer = AutoTokenizer.from_pretrained(test_model_path)
model = AutoModelForCausalLM.from_pretrained(test_model_path, dtype="bfloat16")
model.eval()
print("模型加载完成 ✓")

In [ ]:
# 测试翻译效果
test_cases = [
    "def add(a, b):\n    return a + b",
    "for i in range(10):\n    print(i)",
    "if x > 0:\n    print('positive')\nelse:\n    print('negative')",
    "def factorial(n):\n    if n <= 1:\n        return 1\n    return n * factorial(n-1)",
    "def bubble_sort(arr):\n    n = len(arr)\n    for i in range(n):\n        for j in range(0, n-i-1):\n            if arr[j] > arr[j+1]:\n                arr[j], arr[j+1] = arr[j+1], arr[j]\n    return arr",
]

instruction = "将以下Python代码翻译为光明v3.2代码。"

for py_code in test_cases:
    prompt = f"{instruction}\n\n```python\n{py_code}\n```"
    inputs = tokenizer(prompt, return_tensors="pd")
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"Python: {py_code[:60]}...")
    print(f"光明:   {result[len(prompt):].strip()[:100]}")
    print()

## 第 8 步：（可选）转换为 GGUF 格式

GGUF 格式可用于 llama.cpp 部署，在 CPU 上也能高效推理。

ERNIE-4.5-0.3B 格式特殊，需要自定义写入器。
详细教程：https://aistudio.baidu.com/projectdetail/9749867

In [ ]:
# GGUF 转换（需要自定义写入器，此处仅示意）
# 详细实现请参考 AI Studio 教程

GGUF_OUTPUT = os.path.join(OUTPUT_DIR, 'light_translator.gguf')

print("GGUF 转换步骤:")
print("1. 安装依赖: pip install llama-cpp-python")
print("2. 参考 https://aistudio.baidu.com/projectdetail/9749867 中的自定义写入器")
print(f"3. 将 {MERGED_DIR} 转换为 {GGUF_OUTPUT}")
print(f"4. 验证: llama-cli -m {GGUF_OUTPUT} -p 'def add(a,b): return a+b'")
print()
print("转换完成后，即可在 Ollama / llama.cpp 中使用微调后的模型！")

## 第 9 步：集成到光明 AI 管线

微调后的模型可以嵌入 `light ai` 管线，实现自动化翻译。

In [ ]:
# 管线集成示例
print("光明 AI 管线集成方式:")
print()
print("1. 大模型（Qwen2.5-Coder-7B）根据需求生成 Python 代码")
print("   light ai generate '写一个冒泡排序' --model-size medium")
print()
print("2. 微调后的 0.3B 模型将 Python 翻译为光明")
print("   （此处加载的模型）")
print()
print("3. light ai check 验证光明代码")
print("   light ai check output.light --run")
print()
print("完整闭环：需求 → Python → 光明 → 验证 → 修复")